## Exploratory Data Analysis for the Vancouver Non-Market Housing Dashboard

### Loading and Checking Data

In [24]:
import pandas as pd
import altair as alt
import json

In [25]:
housing = pd.read_csv(
    "../data/raw/non-market-housing.csv",
    sep=";",
    dtype={"Project Status": "category", "Occupancy Year": "Int64"}
)
housing

,Index Number,Name,Address,Project Status,Occupancy Year,Operator,Clientele- Families,Clientele - Seniors,Clientele - Other,Design - Accessible 1BR,...,Design - Adaptable 3BR,Design - Adaptable 4BR,Design - Standard 1BR,Design - Standard 2BR,Design - Standard 3BR,Design - Standard 4BR,Design - Standard Studio,Design - Standard Room,URL,Geom
0,6,Helen's Court Co-op,2137 W 1st Ave,Completed,1984,Helen's Court Co-op Housing Association,35,0,9,2.0,...,0.0,0.0,7.0,22.0,13.0,0.0,0.0,0.0,https://app.vancouver.ca/NonMarketHousing_NET/...,"{""coordinates"": [-123.1536261, 49.27104183], ""..."
1,7,Laura Jamieson Co-op,1349 E 2nd Ave,Completed,1987,Laura Jamieson Co-op Housing Association,42,0,5,1.0,...,0.0,0.0,4.0,23.0,18.0,0.0,0.0,0.0,https://app.vancouver.ca/NonMarketHousing_NET/...,"{""coordinates"": [-123.07614157, 49.26900321], ..."
2,8,Westerdale Co-op,1507 E 2nd Ave,Completed,1984,Westerdale Co-op Housing Association,10,0,9,4.0,...,0.0,0.0,5.0,6.0,2.0,0.0,0.0,0.0,https://app.vancouver.ca/NonMarketHousing_NET/...,"{""coordinates"": [-123.07304966, 49.26896961], ..."
3,18,Vancouver Native (4th Ave),1560 E 4th Ave,Completed,1989,BC Indigenous Housing Society,20,10,1,1.0,...,0.0,0.0,10.0,11.0,4.0,5.0,0.0,0.0,https://app.vancouver.ca/NonMarketHousing_NET/...,"{""coordinates"": [-123.0718679, 49.2666326], ""t..."
4,20,Northern Way Co-op,675 E 5th Ave,Completed,1985,Northern Way Co-op Housing Association,44,0,16,3.0,...,0.0,0.0,13.0,24.0,19.0,1.0,0.0,0.0,https://app.vancouver.ca/NonMarketHousing_NET/...,"{""coordinates"": [-123.09065341, 49.26639799], ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
636,929,NaN,998 E 19th Ave,Approved,<NA>,NaN,0,0,105,22.0,...,NaN,NaN,37.0,26.0,12.0,NaN,30.0,NaN,NaN,"{""coordinates"": [-123.08438908, 49.25336844], ..."
637,934,NaN,1710-1730 E Pender St,Approved,<NA>,Lu'Ma Native Housing Society,71,0,120,NaN,...,NaN,NaN,120.0,38.0,28.0,5.0,NaN,NaN,NaN,"{""coordinates"": [-123.07002235, 49.28001135], ..."
638,948,Brennan's Place,545 E Cordova,Completed,2024,Lookout Housing and Health Society,0,0,20,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,20.0,NaN,https://app.vancouver.ca/NonMarketHousing_NET/...,"{""coordinates"": [-123.0923813, 49.28240033], ""..."
639,989,Murray Hotel,1119 Hornby,Completed,2017,Atira Women’s Resource Society,0,0,95,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,95.0,https://app.vancouver.ca/NonMarketHousing_NET/...,"{""coordinates"": [-123.12750154, 49.27940628], ..."


#### Null Values

In [26]:
housing.isnull().sum()

Index Number                    0
Name                           12
Address                         0
Project Status                  0
Occupancy Year                 63
Operator                       27
Clientele- Families             0
Clientele - Seniors             0
Clientele - Other               0
Design - Accessible 1BR       126
Design - Accessible 2BR       140
Design - Accessible 3BR       153
Design - Accessible 4BR       159
Design - Accessible Studio    129
Design - Accessible Room      156
Design - Adaptable 1BR        155
Design - Adaptable 2BR        158
Design - Adaptable 3BR        159
Design - Adaptable 4BR        160
Design - Standard 1BR          63
Design - Standard 2BR          66
Design - Standard 3BR          77
Design - Standard 4BR         133
Design - Standard Studio       43
Design - Standard Room        107
URL                            50
Geom                           19
dtype: int64

From above, we see that there are null values in the dataset. This is expected in some cases; occupancy year will naturally be null if there are no current occupants in the building, and name, operator, and URL may not currently be available depending on the current stage of the project.

In [27]:
null_cols = [
    "Name",
    "Operator",
    "URL"
]

housing[(housing[null_cols].isnull().any(axis=1)) & (housing["Occupancy Year"].notnull())]

,Index Number,Name,Address,Project Status,Occupancy Year,Operator,Clientele- Families,Clientele - Seniors,Clientele - Other,Design - Accessible 1BR,...,Design - Adaptable 3BR,Design - Adaptable 4BR,Design - Standard 1BR,Design - Standard 2BR,Design - Standard 3BR,Design - Standard 4BR,Design - Standard Studio,Design - Standard Room,URL,Geom


For the most part, null values in the columns related to Design are likely equivalent to 0; we see below that with the exception of one project, all other projects have some design column that is not null. As the one project without any values in any of the design columns has a status of proposed, it is possible that information on the exact design is not currently available.

For this reason, with the exception of the one project, we impute null values in the design columns with zero.

In [28]:
design_cols = [col for col in housing.columns if col.startswith("Design")]
housing[housing[design_cols].isnull().all(axis=1)]

,Index Number,Name,Address,Project Status,Occupancy Year,Operator,Clientele- Families,Clientele - Seniors,Clientele - Other,Design - Accessible 1BR,...,Design - Adaptable 3BR,Design - Adaptable 4BR,Design - Standard 1BR,Design - Standard 2BR,Design - Standard 3BR,Design - Standard 4BR,Design - Standard Studio,Design - Standard Room,URL,Geom
635,926,Shawn Oaks,5505 Oak St,Proposed,<NA>,Affordable Housing Societies,63,0,117,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""coordinates"": [-123.12901006, 49.23571799], ..."


In [29]:
# impute missing values in design columns with 0
mask = ~housing[design_cols].isnull().all(axis=1)
housing[design_cols] = housing[design_cols].where(~mask, housing[design_cols].fillna(0))

This leaves the null values in Geom. There is no reason for Geom to be empty, since all projects have an address. It should be possible to obtain the coordinates from the address in Python using packages such as `geopy`, but for the time being, we will leave them empty.

#### Data Types

In [34]:
housing.dtypes

Index Number                     int64
Name                               str
Address                            str
Project Status                category
Occupancy Year                   Int64
Operator                           str
Clientele- Families              int64
Clientele - Seniors              int64
Clientele - Other                int64
Design - Accessible 1BR          Int64
Design - Accessible 2BR          Int64
Design - Accessible 3BR          Int64
Design - Accessible 4BR          Int64
Design - Accessible Studio       Int64
Design - Accessible Room         Int64
Design - Adaptable 1BR           Int64
Design - Adaptable 2BR           Int64
Design - Adaptable 3BR           Int64
Design - Adaptable 4BR           Int64
Design - Standard 1BR            Int64
Design - Standard 2BR            Int64
Design - Standard 3BR            Int64
Design - Standard 4BR            Int64
Design - Standard Studio         Int64
Design - Standard Room           Int64
URL                      

In [31]:
# convert design columns to integer types
housing[design_cols] = housing[design_cols].astype("Int64")

In [ ]:

housing["coordinates"] = housing["Geom"].apply(
    lambda x: json.loads(x)["coordinates"] if pd.notnull(x) else None
)

housing = housing.drop("Geom", axis=1)

,Index Number,Name,Address,Project Status,Occupancy Year,Operator,Clientele- Families,Clientele - Seniors,Clientele - Other,Design - Accessible 1BR,...,Design - Adaptable 3BR,Design - Adaptable 4BR,Design - Standard 1BR,Design - Standard 2BR,Design - Standard 3BR,Design - Standard 4BR,Design - Standard Studio,Design - Standard Room,URL,coordinates
0,6,Helen's Court Co-op,2137 W 1st Ave,Completed,1984,Helen's Court Co-op Housing Association,35,0,9,2,...,0,0,7,22,13,0,0,0,https://app.vancouver.ca/NonMarketHousing_NET/...,"[-123.1536261, 49.27104183]"
1,7,Laura Jamieson Co-op,1349 E 2nd Ave,Completed,1987,Laura Jamieson Co-op Housing Association,42,0,5,1,...,0,0,4,23,18,0,0,0,https://app.vancouver.ca/NonMarketHousing_NET/...,"[-123.07614157, 49.26900321]"
2,8,Westerdale Co-op,1507 E 2nd Ave,Completed,1984,Westerdale Co-op Housing Association,10,0,9,4,...,0,0,5,6,2,0,0,0,https://app.vancouver.ca/NonMarketHousing_NET/...,"[-123.07304966, 49.26896961]"
3,18,Vancouver Native (4th Ave),1560 E 4th Ave,Completed,1989,BC Indigenous Housing Society,20,10,1,1,...,0,0,10,11,4,5,0,0,https://app.vancouver.ca/NonMarketHousing_NET/...,"[-123.0718679, 49.2666326]"
4,20,Northern Way Co-op,675 E 5th Ave,Completed,1985,Northern Way Co-op Housing Association,44,0,16,3,...,0,0,13,24,19,1,0,0,https://app.vancouver.ca/NonMarketHousing_NET/...,"[-123.09065341, 49.26639799]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
636,929,NaN,998 E 19th Ave,Approved,<NA>,NaN,0,0,105,22,...,0,0,37,26,12,0,30,0,NaN,"[-123.08438908, 49.25336844]"
637,934,NaN,1710-1730 E Pender St,Approved,<NA>,Lu'Ma Native Housing Society,71,0,120,0,...,0,0,120,38,28,5,0,0,NaN,"[-123.07002235, 49.28001135]"
638,948,Brennan's Place,545 E Cordova,Completed,2024,Lookout Housing and Health Society,0,0,20,0,...,0,0,0,0,0,0,20,0,https://app.vancouver.ca/NonMarketHousing_NET/...,"[-123.0923813, 49.28240033]"
639,989,Murray Hotel,1119 Hornby,Completed,2017,Atira Women’s Resource Society,0,0,95,0,...,0,0,0,0,0,0,0,95,https://app.vancouver.ca/NonMarketHousing_NET/...,"[-123.12750154, 49.27940628]"
